In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, accuracy_score, log_loss
from sklearn.metrics import  precision_score, recall_score, classification_report,confusion_matrix, balanced_accuracy_score
import joblib
from pathlib import Path
import sys
import os


In [2]:
def find_project_root():
    """
    Search parent directories for ROOT folder (that contains src folder)`.
    add it to the sys path
    """
    current_path = Path.cwd()  # start from current working directory
    for parent in [current_path] + list(current_path.parents):
        if (parent / 'src').exists():
            #return (parent / 'src').resolve()
            return (parent).resolve()
    raise FileNotFoundError(f"src folder not found in any parent directory.")

# Add root directory to system path
ROOT_DIR =  find_project_root()
print("Project root directory:", ROOT_DIR)
sys.path.append(f"{ROOT_DIR}")

Project root directory: /gpfs/exfel/data/scratch/majida/Noble_Gas_Instability/New_analysis-aug-2025/git_repo_for_paper/VacFault (21-03-2026)


In [3]:
from src.evaluation.evaluate_svm import evaluate_svm

In [4]:
# important parameters from configuration file
from src.config import ROOT_DIR
from src.config import datasets_dir
from src.config import models_dir
from src.config import datasets
from src.config import feature_col
from src.config import ml_models_svm
from src.config import imbalance_learn_methods_svm

In [10]:
# explore results of svm-poly trained using smote pluse cost-sensitive learning

ml_model = [model for model in ml_models_svm if model == 'svm_poly'][0]
imbalance_learn_method = [method for method in imbalance_learn_methods_svm if method=='smote_plus_cost_sensitive'][0]
dataset = [dataset for dataset in datasets if dataset=='deployment_data'][0]

print("ml_model: ",ml_model,
      "imbalance learn method: ",imbalance_learn_method,
      "dataset: ", dataset
    )

ml_model:  svm_poly imbalance learn method:  smote_plus_cost_sensitive dataset:  deployment_data


In [11]:
# Explore results
data_dir = os.path.join(datasets_dir,dataset)
df = pd.read_hdf(f"{data_dir}/df_deployment.h5")
current_model_dir = f"{models_dir}/{ml_model}_{imbalance_learn_method}/"
# Load pipeline parameters
pipeline_name = f"{current_model_dir}pipeline.pkl"
pipeline = joblib.load(pipeline_name)
    
# Separate features vs.labels
X = df[feature_col]
y = df['labels']
y = y.astype(int)

# Evaluate performance
y_pred = pipeline.predict(X)
precision = precision_score(y, y_pred, average=None)
recall = recall_score(y, y_pred, average=None)
f1 = f1_score(y, y_pred, average=None)

In [25]:
# See total number of examples classified as "Fault"
X_pred = X.copy()
X_pred['predictions'] = y_pred
print(X_pred['predictions'] .value_counts())
n = X_pred['predictions'] .value_counts().loc[1]
print("total number of samples classified as 'Fault': ",n)

0    956
1     53
Name: predictions, dtype: int64
total number of samples classified as 'Fault':  53


In [27]:
X_predicted_as_fault = X_pred[X_pred['predictions']==1]
#print(X_predicted_as_fault.shape)
# True faults
print(y.loc[X_predicted_as_fault.index].value_counts())
k = y.loc[X_predicted_as_fault.index].value_counts().loc[1]
print("Number of true faults in samples classified as 'Fault': ",k)

1    41
0    12
Name: labels, dtype: int64
Number of true faults in samples classified as 'Fault':  41


In [20]:
print(classification_report(y,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       963
           1       0.77      0.89      0.83        46

    accuracy                           0.98      1009
   macro avg       0.88      0.94      0.91      1009
weighted avg       0.98      0.98      0.98      1009

